In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D
from scipy.interpolate import interp1d
import pandas as pd
import sncosmo
from scipy import stats
from scipy.special import expit
from nested_pandas import read_parquet
from joblib import Parallel, delayed
from scipy.stats import chi2
import cloudpickle as pickle
from astropy.cosmology import FlatLambdaCDM
from ndtest import ks2d2s

from lightcurvelynx.astro_utils.mag_flux import mag2flux,flux2mag
from lightcurvelynx.utils.plotting import plot_lightcurves
from lightcurvelynx.graph_state import GraphState
from lightcurvelynx.simulate import compute_single_noise_free_lightcurve

from lightcurvelynx import _LIGHTCURVELYNX_BASE_DATA_DIR

from lightcurvelynx.validation.lcfit import fit_single_lc

from utils.plotting_utils import (plot_snr_distr, 
                                plot_logflux_vs_logfluxerr,
                                plot_logflux_vs_logfluxerr_corner,
                                plot_logmaxflux_vs_logmaxfluxerr,
                                plot_logmaxflux_vs_logmaxfluxerr_corner,
                                convert_flux_to_mag,
                                plot_mag_vs_magerr,
                                convert_flux_to_njy,
                                plot_flux_vs_fluxerr,
                                )

In [ ]:
globalhostdata = pd.read_csv('ztfsniadr2/tables/globalhost_data.csv')
localhostdata = pd.read_csv('ztfsniadr2/tables/localhost_data.csv')
sndata = pd.read_csv('ztfsniadr2/tables/snia_data.csv')
data = pd.merge(sndata,globalhostdata,on='ztfname')

In [ ]:
lcdata = read_parquet('data/ztfsniadr2.parquet')

In [ ]:
lightcurves = read_parquet('results/lightcurves.parquet')

In [ ]:
lightcurves = lightcurves.rename(columns={"lightcurve":"lc"})

In [ ]:
lightcurves.head()

In [ ]:
lightcurves['params'][0].keys()

In [ ]:
print("All sample simulated: nsn=", len(lightcurves))
# drop saturation
lightcurves_after_drop_sat = lightcurves.query("lc.is_saturated==False").dropna()
print("After dropping saturation: nsn=", len(lightcurves_after_drop_sat))
lightcurves_after_detection = lightcurves_after_drop_sat.query("lc.detection_flag == True").dropna()
print("After applying detection: nsn=", len(lightcurves_after_detection))
lightcurves_after_spec_selection = lightcurves.loc[lightcurves['pass_spec_selection']]
print("After spectroscopic selection: nsn=", len(lightcurves_after_spec_selection))
lightcurves_after_quality_cut = lightcurves.loc[lightcurves['pass_quality_cuts']]
print("After quality cuts: nsn=", len(lightcurves_after_quality_cut))

In [ ]:
with open("results/saved_model_and_passband.pkl", "rb") as file:  
    lynx_model, passband_group = pickle.load(file)

In [ ]:
sncosmo_modelname = "salt3"
random_ids = lightcurves_after_quality_cut.id.sample(1).values
colormap = {'g':'g',
            'r':'r',
            'i':'purple',}
print(random_ids)

for random_id in random_ids:
    # Extract the row for this object.
    # lc = lightcurves.loc[lightcurves.id==random_id]
    lc = lightcurves_after_quality_cut.loc[lightcurves_after_quality_cut.id==random_id]
    lc = lc.query("lc.detection_flag==True")
    
    state = GraphState.from_dict(lc.iloc[0]["params"])
    noise_free_lcs = compute_single_noise_free_lightcurve(
        lynx_model,
        state,
        passband_group,
        rest_frame_phase_min=-20.0, 
        rest_frame_phase_max=50.0, 
        rest_frame_phase_step=0.5, 
    )
    
    if lc["nobs"].values[0] > 0:
        # Unpack the nested columns (filters, mjd, flux, and flux error).
        lc_filters = np.asarray(lc["lc.filter"], dtype=str)
        lc_mjd = np.asarray(lc["lc.mjd"], dtype=float)
        lc_flux = np.asarray(lc["lc.flux"], dtype=float)
        lc_fluxerr = np.asarray(lc["lc.fluxerr"], dtype=float)
        
        lc_mag = -2.5*np.log10(lc_flux) + 31.4
        lc_magerr = np.absolute(1.086*lc_fluxerr/lc_flux)
        
        plot_lightcurves(
            fluxes=lc_flux,
            times=lc_mjd,
            fluxerrs=lc_fluxerr,
            filters=lc_filters,
            colormap=colormap,
            underlying_model=noise_free_lcs,
        )
        plt.ylabel('Flux (nJy)')
        # plt.title(f"redshift={lc['z']},mwebv={lc['params'].values[0]['mwext.ebv']}")
        plt.title(f"redshift={lc['z'].values[0]},x1={lc['params'].values[0]['source.x1']},c={lc['params'].values[0]['source.c']}")
        for par in ['x0','x1','c']:
            print(par,lc['params'].values[0][f'source.{par}'])
        print('z',lc['z'].values[0])
        print('mwebv',lc['params'].values[0]['mwext.ebv'])
        
        model = sncosmo.Model(source=sncosmo_modelname,
                      effects=[sncosmo.F99Dust()],
                      effect_names=['mw'],
                      effect_frames=['obs'])
        pardict = {}
        for p in ['x0','x1','c','t0']:
            pardict[p] = lc['params'].values[0][f'source.{p}']
        pardict['mwebv'] = lc['params'].values[0]['mwext.ebv']
        pardict['z'] = lc['z'].values[0]
        model.update(pardict)
        obstime = noise_free_lcs["times"][1:-1]
        sncosmo_flux = {}
        lynx_flux = {}
        mjd = {}
                        
        # for b in 'gri':
        #     lynx_pb = passband_group.passbands[f'ZTF_{b}']
        #     wave = lynx_pb._loaded_table[:,0] #lynx_pb.processed_transmission_table[:,0]
        #     trans = lynx_pb._loaded_table[:,1] #lynx_pb.processed_transmission_table[:,1]
        #     pb = sncosmo.Bandpass(wave,trans)
        #     # pb = sncosmo.get_bandpass(f"ztf{b}")
        #     sncosmo_flux[b] = model.bandflux(pb,obstime,zp=31.4,zpsys='ab')
        #     plt.plot(obstime,sncosmo_flux[b],label=b,ls=':',color=colormap[b])
        #     lynx_flux[b] = noise_free_lcs[b][1:-1]
            
        plt.legend()
        plt.show()

        # for b in 'gri':
        #     plt.plot(obstime, lynx_flux[b] / sncosmo_flux[b],label=b)
        # plt.xlabel("time")
        # plt.ylabel("lynx_bandflux/sncosmo_bandflux")
        # plt.legend()
        
        # plot_lightcurves(
        #     fluxes=lc_mag,
        #     times=lc_mjd,
        #     fluxerrs=lc_magerr,
        #     filters=lc_filters,
        # )
        # plt.ylabel('Mag')
        # plt.ylim(plt.ylim()[::-1])
        # plt.show()

In [ ]:
saltpars = pd.read_csv('results/salt3fit_results.csv')
len(saltpars)

In [ ]:
saltpars.columns

In [ ]:
## make cuts based on salt parameters
saltpar_cuts = (saltpars.x1 > -3) & (saltpars.x1 < 3)
print("x1 > -3 & x1 < 3: ",np.sum(saltpar_cuts))
saltpar_cuts &= (saltpars.c > -0.2) & (saltpars.c < 0.8)
print("c > -0.2 & x1 < 0.8: ",np.sum(saltpar_cuts))
saltpar_cuts &= saltpars.t0_err < 1
print("sigma_t0 < 1: ",np.sum(saltpar_cuts))
saltpar_cuts &= saltpars.x1_err < 1
print("sigma_x1 < 1: ",np.sum(saltpar_cuts))
saltpar_cuts &= saltpars.c_err < 0.1
print("sigma_c < 0.1: ",np.sum(saltpar_cuts))

reduced = saltpars.chisq / saltpars.ndof
p_value = 1 - chi2.cdf(saltpars.chisq, df=saltpars.ndof)

saltpar_cuts &= p_value > 1e-7
print("fitprob > 1e-7: ",np.sum(saltpar_cuts))


In [ ]:
bins=np.linspace(0,0.21,30)
lc_to_plot = lightcurves_after_spec_selection

plt.hist(lc_to_plot['z'],bins=bins,alpha=0.3,density=True,label='sim after spec selection',color='b')
plt.hist(data.redshift,bins=bins,alpha=0.3,density=True,label='ztf sn data release',color='r')

ks = stats.ks_2samp(lc_to_plot['z'],data.redshift)
print(ks)
plt.legend()
plt.show()
# plt.savefig('paper_figs/z_distr.png')

In [ ]:
bins=np.linspace(0,0.21,30)
lc_to_plot = lightcurves_after_quality_cut

plt.hist(lc_to_plot['z'],bins=bins,alpha=0.3,density=True,
         histtype='step',label='sim after quality cuts',color='b',lw=2)
plt.hist(data.loc[data.lccoverage_flag==1].redshift,bins=bins,alpha=0.3,density=True,
         histtype='step',label='ztf sn after quality cuts',color='r', lw=2)
ks = stats.ks_2samp(lc_to_plot['z'],data.loc[data.lccoverage_flag==1].redshift)
print(ks)

plt.legend()
plt.xlabel('Redshift')
plt.ylabel('Count (Normalized)')
plt.savefig('paper_figs/z_distr.png')

In [ ]:
# data quality flags

# from https://github.com/ZwickyTransientFacility/ztfcosmo/blob/260e136be064708e1238719880ce18844027422e/ztfcosmo/lightcurve.py#L111
# flagout: [list of int or string]
#     flag == 0 means all good, but may not be detected:
    
#     0: no warning 
#     1: flux_err==0 Remove unphysical errors 
#     2: chi2dof>3: Remove extreme outliers 
#     4: cloudy>1: BTS cut 
#     8: infobits>0: BTS cut 
#     16: mag_lim<19.3: Cut applied in Dhawan 2021 
#     32: seeing>3: Cut applied in Dhawan 2021 
#     64: fieldid>879: Recommended IPAC cut 
#     128: moonilf>0.5: Recommended IPAC cut 
#     256: has_baseline>1: Has a valid baseline correction 
#     512: airmass>2: Recommended IPAC cut 
#     1024: flux/flux_err>=5: Nominal detection

In [ ]:
# ztf sndr2 flags
# sn_type SNIa classification 
# sub_type sub classification if any. 
# lccoverage_flag passes the good sampling cut (bool,Table1) 
# fitquality_flag passes all other Basic cuts (bool,Table1)

In [ ]:
def filter_flags(lc_flag, flags_to_exclude=[], flags_to_include=[]):
    pass_filter = True
    if len(flags_to_include)>0:
        pass_filter &= np.all([lc_flag & flag != 0 for flag in flags_to_include])
    if len(flags_to_exclude)>0:
        pass_filter &= np.all([lc_flag & flag == 0 for flag in flags_to_exclude])
    return pass_filter

In [ ]:
#calculate this first
lcdata["lc.pass_flag_filter"] = lcdata["lc.flag"].apply(filter_flags,flags_to_exclude=[1,2,4,8,16,32],flags_to_include=[])
lcdata["lc.pass_detection"] = lcdata["lc.flag"].apply(filter_flags,flags_to_exclude=[1,2,4,8,16,32],flags_to_include=[1024])

In [ ]:
def calculate_median_cadence(mjd):
    mjd = np.sort(mjd)
    return np.median(np.diff(mjd))

In [ ]:
#caculate nobs, ndetections

for f in "gr":
    lc_to_show = lightcurves_after_spec_selection.dropna(subset="lc")
    data_to_show = lcdata.query("lc.pass_flag_filter == True").dropna(subset="lc")
    t_min_obslog,t_max_obslog = 58288.171875, 59273.5546875
    data_to_show = data_to_show.query(f"lc.mjd> {t_min_obslog} & lc.mjd < {t_max_obslog}")

    lc_to_show["lc"] = lc_to_show["lc"].nest[lc_to_show["lc.filter"] == f]
    data_to_show["lc"] = data_to_show["lc"].nest[data_to_show["lc.filter"] == "ztf"+f]

    nobs_sim = lc_to_show['lc.mjd'].count()
    nobs_data = data_to_show['lc.mjd'].count()
    
    ndet_sim = lc_to_show.query("lc.detection_flag == True")['lc.mjd'].count()
    ndet_data = data_to_show.query("lc.pass_detection == True")['lc.mjd'].count()
    
    avg_nobs_sim = nobs_sim/len(lc_to_show)
    avg_nobs_data = nobs_data/len(data_to_show)
    avg_ndet_sim = ndet_sim/len(lc_to_show)
    avg_ndet_data = ndet_data/len(data_to_show)
    
    avg_cadence_sim = lc_to_show.reduce(calculate_median_cadence,"lc.mjd")
    avg_cadence_data = data_to_show.reduce(calculate_median_cadence,"lc.mjd")
    
    avg_cadence_sim_det = lc_to_show.query("lc.detection_flag==True").reduce(calculate_median_cadence,"lc.mjd")
    avg_cadence_data_det = data_to_show.query("lc.pass_detection==True").reduce(calculate_median_cadence,"lc.mjd")

    print(f"filter = {f}")
    print("     |Nevent|Nobs    | Ndetection | Avg Nobs      | Avg Ndet   |Avg cadence  |Avg cadence detection ")

    print(f" sim |{len(lc_to_show)} | {nobs_sim} | {ndet_sim} | {avg_nobs_sim}  | {avg_ndet_sim}  |{avg_cadence_sim.mean()[0]} | {avg_cadence_sim_det.mean()[0]}")
    print(f" data|{len(data_to_show)} | {nobs_data} | {ndet_data} | {avg_nobs_data} | {avg_ndet_data}  |{avg_cadence_data.mean()[0]} |{avg_cadence_data_det.mean()[0]}")


In [ ]:
avg_cadence_sim.hist(bins=np.linspace(0,5))
avg_cadence_data.hist(bins=np.linspace(0,5))

In [ ]:
avg_cadence_sim_det.hist(bins=np.linspace(0,5))
avg_cadence_data_det.hist(bins=np.linspace(0,5))

In [ ]:
lc_to_plot = lightcurves.loc[lightcurves_after_spec_selection.index]
lcdata_plot = lcdata
lcdata_plot['lc.snr'] = lcdata_plot['lc.flux']/lcdata_plot['lc.flux_err']

bins = np.linspace(0,5,30)

plot_snr_distr([lc_to_plot, lcdata_plot],bins=bins,alpha=0.3,density=True,labels=['sim','data'])
plt.xlabel('SNR')
plt.ylabel('Count (Normalized)')
plt.title('All observations')
plt.savefig('paper_figs/snr_allobs.png')

In [ ]:
lc_to_plot = lightcurves_after_detection.loc[lightcurves_after_spec_selection.index]

# lcdata["lc.pass_flag_filter"] = lcdata["lc.flag"].apply(filter_flags,flags_to_exclude=[1,2,4,8,16,32,64,128,256,512],flags_to_include=[1024])
lcdata_plot = lcdata.query("lc.pass_flag_filter == True").dropna(subset="lc")
lcdata_plot['lc.snr'] = lcdata_plot['lc.flux']/lcdata_plot['lc.flux_err']
lcdata_plot = lcdata_plot.query("lc.snr > 5").dropna(subset="lc")

bins = np.linspace(0,100,30)

plot_snr_distr([lc_to_plot, lcdata_plot],bins=bins,alpha=0.3,density=True,labels=['sim','data'])
plt.xlabel('SNR')
plt.ylabel('Count (Normalized)')
plt.title('All Detections')
plt.savefig('paper_figs/snr_alldetection.png')

In [ ]:
lc_to_plot = lightcurves_after_detection.loc[lightcurves_after_spec_selection.index]

lcdata_plot = lcdata.query("lc.pass_flag_filter == True").dropna(subset="lc")
lcdata_plot['lc.snr'] = lcdata_plot['lc.flux']/lcdata_plot['lc.flux_err']
lcdata_plot = lcdata_plot.query("lc.snr > 5").dropna(subset="lc")

lc_to_plot['lc.mag'],lc_to_plot['lc.magerr'] = convert_flux_to_mag(lc_to_plot['lc.flux'],lc_to_plot['lc.fluxerr'],zp=31.4)
lcdata_plot['lc.mag'],lcdata_plot['lc.magerr'] = convert_flux_to_mag(lcdata_plot['lc.flux'],lcdata_plot['lc.flux_err'],zp=30.)

plot_mag_vs_magerr(lc_to_plot,lcdata_plot)

In [ ]:
idx = lc_to_plot["lc.filter"] == 'r'
plt.plot(lc_to_plot["lc.mag"][idx],lc_to_plot["lc.magerr"][idx]/(2.5/np.log(10.)),'.',alpha=0.2)
plt.xlim((13.5,21.5))
plt.show()
idx = lcdata_plot["lc.filter"] == 'ztfr'
plt.plot(lcdata_plot["lc.mag"][idx],lcdata_plot["lc.magerr"][idx]/(2.5/np.log(10.)),'.',alpha=0.1)
plt.xlim((13.5,21.5))


In [ ]:
plot_flux_vs_fluxerr(lc_to_plot,lcdata_plot)

In [ ]:
# check about the outliers in data
maglim = mag2flux(12)
idx = lcdata_plot.query(f'lc.flux > {maglim}').dropna(subset='lc').index
outlier = lcdata_plot.loc[idx]
for i in range(0,np.min([10,len(outlier)])):
    plt.plot(outlier.iloc[i]['lc']['mjd'],flux2mag(outlier.iloc[i]['lc']['flux']),'o')
    plt.ylim(plt.ylim()[::-1])
    plt.show()
    print(outlier.iloc[i]['ztfname'])
    print(outlier.iloc[i]['lc'])

In [ ]:
lc_to_plot = lightcurves_after_detection.loc[lightcurves_after_spec_selection.index]

lcdata_plot = lcdata.query("lc.pass_flag_filter == True").dropna(subset="lc")
lcdata_plot['lc.snr'] = lcdata_plot['lc.flux']/lcdata_plot['lc.flux_err']
lcdata_plot = lcdata_plot.query("lc.snr > 5").dropna(subset="lc")

plot_logflux_vs_logfluxerr(lc_to_plot, lcdata_plot)

In [ ]:
plot_logflux_vs_logfluxerr_corner(lc_to_plot, lcdata_plot, smooth_sigma=0.6)
plt.savefig('paper_figs/logflux_contour.png')

In [ ]:
lc_to_plot = lightcurves_after_detection.loc[lightcurves_after_spec_selection.index]

lcdata_plot = lcdata.query("lc.pass_flag_filter == True").dropna(subset="lc")
lcdata_plot['lc.snr'] = lcdata_plot['lc.flux']/lcdata_plot['lc.flux_err']
lcdata_plot = lcdata_plot.query("lc.snr > 5").dropna(subset="lc")

plot_logmaxflux_vs_logmaxfluxerr(lc_to_plot,lcdata_plot)

In [ ]:
plot_logmaxflux_vs_logmaxfluxerr_corner(lc_to_plot,lcdata_plot,n_levels=10,smooth_sigma=0.8)
plt.savefig('paper_figs/logmaxflux_contour.png')

In [ ]:
lc_to_plot = lightcurves_after_detection.loc[lightcurves_after_spec_selection.index]

lcdata_plot = lcdata.query("lc.pass_flag_filter == True").dropna(subset="lc")
lcdata_plot['lc.snr'] = lcdata_plot['lc.flux']/lcdata_plot['lc.flux_err']
lcdata_plot = lcdata_plot.query("lc.snr > 5").dropna(subset="lc")

lc_to_plot['lc.mag'],lc_to_plot['lc.magerr'] = convert_flux_to_mag(lc_to_plot['lc.flux'],lc_to_plot['lc.fluxerr'],zp=31.4)
lcdata_plot['lc.mag'],lcdata_plot['lc.magerr'] = convert_flux_to_mag(lcdata_plot['lc.flux'],lcdata_plot['lc.flux_err'],zp=30.)

lc_to_plot = lc_to_plot.query("lc.mag > 18 & lc.mag < 18.5").dropna(subset="lc")
lcdata_plot = lcdata_plot.query("lc.mag > 18 & lc.mag < 18.5").dropna(subset="lc")

plot_logflux_vs_logfluxerr(lc_to_plot, lcdata_plot)

In [ ]:
sim_all_x1 = lightcurves['source_x1'].dropna()
sim_all_c = lightcurves['source_c'].dropna()

In [ ]:
lc_to_plot = lightcurves_after_spec_selection
sim_x1 = lc_to_plot['source_x1'].dropna()
sim_c = lc_to_plot['source_c'].dropna()

In [ ]:
def expo(x, a, b, c):
    x = np.asarray(x)
    y = a * np.exp(b * x) + c
    return y

In [ ]:
fx1_parr = np.loadtxt('data/ztf_selection_func_x1.txt')
fc_parr = np.loadtxt('data/ztf_selection_func_c.txt')
fx1_func = lambda x1: expo(x1, *fx1_parr)
fc_func = lambda c: expo(c, *fc_parr)

In [ ]:
bins = np.linspace(-5,5,10)
x1before,bin_edges,_ = stats.binned_statistic(sim_all_x1,np.ones(len(lightcurves)), statistic='sum', bins=bins)
x1after,bin_edges,_ =  stats.binned_statistic(sim_x1, np.ones(len(lc_to_plot)), statistic='sum', bins=bins)
norm = 1./interp1d((bin_edges[:-1] + bin_edges[1:])/2.,x1after/x1before)(0)
err = np.sqrt(x1after**2/x1before**3 + x1after/x1before**2)
plt.errorbar((bin_edges[:-1] + bin_edges[1:])/2.,x1after/x1before*norm,yerr = err*norm, fmt='o')
xplot = np.linspace(-5,5,50)
plt.plot(xplot,fx1_func(xplot)/fx1_func(0))
plt.xlabel('x1')
plt.ylabel('Relative Selection Ratio')
plt.savefig('paper_figs/x1_selection.png')

In [ ]:
bins = np.linspace(-0.5,1,10)
cbefore,bin_edges,_ = stats.binned_statistic(sim_all_c,np.ones(len(lightcurves)), statistic='sum', bins=bins)
cafter,bin_edges,_ =  stats.binned_statistic(sim_c, np.ones(len(lc_to_plot)), statistic='sum', bins=bins)
norm = 1./interp1d((bin_edges[:-1] + bin_edges[1:])/2.,cafter/cbefore)(0)
err = np.sqrt(cafter**2/cbefore**3 + cafter/cbefore**2)
plt.errorbar((bin_edges[:-1] + bin_edges[1:])/2.,cafter/cbefore*norm,yerr = err*norm, fmt='o')
xplot = np.linspace(-0.5,1,50)
plt.plot(xplot,fc_func(xplot)/fc_func(0))
plt.xlabel('c')
plt.ylabel('Relative Selection Ratio')
plt.savefig('paper_figs/c_selection.png')

In [ ]:
result_df = saltpars
idx = result_df.ndof > 0
result_df = result_df[idx]
plt.hist(result_df[idx].chisq/result_df[idx].ndof,bins=np.linspace(0,10,20))
plt.title('reduced chisq')

In [ ]:
lc_to_fit = lightcurves

In [ ]:
x1 = result_df['x1']
sim_x1 = [lc_to_fit.loc[i]['params']['source.x1'] for i in result_df.id]
plt.hist(x1-sim_x1,bins=np.linspace(-0.5,0.5,30))
plt.title('x1-sim_x1')
plt.xlabel('x1')
plt.ylabel('Number')
plt.savefig('paper_figs/x1diff.png')

In [ ]:
c = result_df['c']
sim_c = [lc_to_fit.loc[i]['params']['source.c'] for i in result_df.id]
plt.hist(c-sim_c,bins=np.linspace(-0.05,0.05,30))
plt.title('c-sim_c')
plt.xlabel('c')
plt.ylabel('Number')
plt.savefig('paper_figs/cdiff.png')

In [ ]:
x0 = result_df['x0']
sim_x0 = [lc_to_fit.loc[i]['params']['source.x0'] for i in result_df.id]
plt.hist(x0-sim_x0,bins=np.linspace(-1.e-3,1.e-3))
plt.title('x0-sim_x0')

In [ ]:
x0 = result_df['t0']
sim_x0 = [lc_to_fit.loc[i]['params']['source.t0'] for i in result_df.id]
plt.hist(np.log10(x0/sim_x0),bins=np.linspace(-1.e-4,1e-4))
plt.title('log(x0)-log(sim_x0)')

In [ ]:
bins=np.linspace(-5,5,30)

plt.hist(x1,bins=bins,density=True,label='fit_to_sim',histtype = 'step',lw = 2, color='blue',alpha=0.3)
plt.hist(data.x1,bins=bins,density=True,label='data',histtype = 'step',lw = 2, color='red',alpha=0.3)
plt.legend()
plt.xlabel('x1')
plt.ylabel('Count (Normalized)')

ks = stats.ks_2samp(x1,data.x1.dropna())
print(ks)
plt.savefig('paper_figs/x1_distr.png')

In [ ]:
bins=np.linspace(-0.5,1,30)
plt.hist(c,bins=bins,density=True,label='fit_to_sim',histtype = 'step',lw = 2, color='blue',alpha=0.3)
plt.hist(data.c,bins=bins,density=True,label='data',histtype = 'step',lw = 2, color='red',alpha=0.3)
plt.legend()
# plt.title('c')
plt.xlabel('c')
plt.ylabel('Count (Normalized)')

ks = stats.ks_2samp(c,data.c.dropna())
print(ks)
plt.savefig('paper_figs/c_distr.png')

In [ ]:
hostmass = lightcurves_after_quality_cut['host_hostmass']
bins = np.linspace(5,13,30)
plt.hist(hostmass,bins=bins,density=True,label='sim',histtype = 'step',lw = 2, color='blue',alpha=0.3)
plt.hist(data.loc[data.lccoverage_flag == 1].mass,bins=bins,density=True,label='data',histtype = 'step',lw = 2, color='red',alpha=0.3)
plt.legend()

ks = stats.ks_2samp(x1,data.x1.dropna())
print(ks)

plt.xlabel('host mass')
plt.ylabel('Count(Normalized)')
plt.savefig('paper_figs/hostmass_distr.png')

In [ ]:
idx = saltpar_cuts
data_idx = (data.fitquality_flag == True) & (data.lccoverage_flag == True)
print(np.sum(idx))
print(np.sum(data_idx))

fig = plt.figure()
plt.subplot(1,1,1)
plt.plot(data[data_idx].mass,data[data_idx].x1,'o',label='ztf sn data release',alpha=0.05,color='red')
plt.plot(np.array(hostmass)[idx],np.array(x1)[idx],'o',label='sim',alpha=0.05,color='blue')
plt.xlabel('hostmass')
plt.ylabel('x1')
plt.legend()
plt.title('x1 vs host')
# plt.show()

# plt.subplot(2,1,2)
bins_x1 = np.linspace(-4,4,15)
bins_hostmass = np.linspace(6,13,15)

data_plot = data[data_idx]
sim_host = np.array(hostmass)[idx]
sim_x1 = np.array(x1)[idx]

binwidth_host = bins_hostmass[1]-bins_hostmass[0]
binwidth_x1 = bins_x1[1]-bins_x1[0]

sim_count,sim_x_edges,sim_y_edges, _ = stats.binned_statistic_2d(sim_host,sim_x1,
                                             np.ones(len(sim_host)),
                                             statistic='sum',bins=[bins_hostmass,bins_x1])
data_count,data_x_edges,data_y_edges,_ = stats.binned_statistic_2d(data_plot.mass,data_plot.x1,
                                         np.ones(len(data_plot.mass)),
                                         statistic='sum',bins=[bins_hostmass,bins_x1])
sim_x = 0.5* (sim_x_edges[:-1]+sim_x_edges[1:])
sim_y = 0.5* (sim_y_edges[:-1]+sim_y_edges[1:])
data_x = 0.5* (data_x_edges[:-1]+data_x_edges[1:])
data_y = 0.5* (data_y_edges[:-1]+data_y_edges[1:])
sim_x_plot,sim_y_plot = np.meshgrid(sim_x, sim_y)
data_x_plot,data_y_plot = np.meshgrid(data_x, data_y)
CS = plt.contour(sim_x_plot.T,sim_y_plot.T,sim_count/np.sum(sim_count)/binwidth_host/binwidth_x1,alpha=0.5,levels=[0.01,0.05,0.1],colors='blue')
CS = plt.contour(data_x_plot.T,data_y_plot.T,data_count/np.sum(data_count)/binwidth_host/binwidth_x1,alpha=0.5,levels=[0.01,0.05,0.1],colors='red')
proxies = [Line2D([],[],color=c) for c in ['blue','red']]
plt.legend(proxies,['sim', 'data'])
plt.xlabel('host mass')
plt.ylabel('x1')
plt.ylim((-4,4))
plt.tight_layout()
fig.savefig('paper_figs/host_x1.png')

ks = ks2d2s(sim_x,sim_y,data_x,data_y)
print(ks)

In [ ]:
# make HD

x1_hd = result_df.loc[saltpar_cuts,"x1"]
c_hd = result_df.loc[saltpar_cuts,"c"]
x0_hd = result_df.loc[saltpar_cuts,"x0"]
z_hd = result_df.loc[saltpar_cuts,"z"]

x1_err = result_df.loc[saltpar_cuts,"x1_err"]
c_err = result_df.loc[saltpar_cuts,"c_err"]
x0_err = result_df.loc[saltpar_cuts,"x0_err"]

mb_hd = -2.5*np.log10(x0_hd) + 10.635
mb_err = 2.5/np.log(10) * (x0_err/x0_hd)

alpha = 0.14
beta = 3.1
Mb = -19.05
mu_hd = mb_hd + alpha*x1_hd - beta*c_hd - Mb
mu_err = np.sqrt(mb_err**2 + alpha**2*x1_err**2 + beta**2*c_err**2 + 0.1**2)

ax1 = plt.subplot(3,1,1)

ax1.errorbar(z_hd, mu_hd, yerr=mu_err,fmt='.',alpha=0.3)

z_cosmo = np.linspace(z_hd.min(),z_hd.max(),50)
cosmo = FlatLambdaCDM(H0=70, Om0=0.3)
mu_cosmo = cosmo.distmod(z_cosmo)
ax1.plot(z_cosmo, mu_cosmo)

plt.xlabel('z')
plt.ylabel('mu')

mures = mu_hd - cosmo.distmod(z_hd).value

ax2 = plt.subplot(3,1,2, sharex=ax1)
ax2.errorbar(z_hd, mures, yerr=mu_err, fmt='.',alpha=0.3)
colors = plt.rcParams["axes.prop_cycle"].by_key()["color"]
ax2.axhline(y=0,color=colors[1])
plt.ylabel('mures')

bins= np.linspace(z_hd.min(),z_hd.max(),15)
mures_mean, bin_edges, _ = stats.binned_statistic(
    z_hd, mures, statistic='mean', bins= bins
)

mures_std, bin_edges, _ = stats.binned_statistic(
    z_hd, mures, statistic='std', bins= bins
)

mures_count, bin_edges, _ = stats.binned_statistic(
    z_hd, mures, statistic='count', bins= bins
) 

mures_err = mures_std / np.sqrt(mures_count)

bin_mean = (bin_edges[:-1] + bin_edges[1:])*0.5
ax2.errorbar(bin_mean, mures_mean, yerr = mures_err,fmt='o')

ax3 = plt.subplot(3,1,3, sharex=ax1)
ax3.errorbar(bin_mean, mures_mean, yerr = mures_err,fmt='o')
ax3.axhline(y=0,color=colors[1])

plt.savefig('paper_figs/HD.png')

In [ ]:
# make HD

data_x1_hd = data[data_idx].x1
data_c_hd = data[data_idx].c
data_x0_hd = data[data_idx].x0
data_z_hd = data[data_idx].redshift

data_x1_err = data[data_idx].x1_err
data_c_err = data[data_idx].c_err
data_x0_err = data[data_idx].x0_err

data_mb_hd = -2.5*np.log10(data_x0_hd) + 10.635
data_mb_err = 2.5/np.log(10) * (data_x0_err/data_x0_hd)

alpha = 0.14
beta = 3.1
Mb = -19.05
data_mu_hd = data_mb_hd + alpha*data_x1_hd - beta*data_c_hd - Mb
data_mu_err = np.sqrt(data_mb_err**2 + alpha**2*data_x1_err**2 + beta**2*data_c_err**2 + 0.1**2)

ax1 = plt.subplot(3,1,1)

ax1.errorbar(data_z_hd, data_mu_hd, yerr=data_mu_err,fmt='.',alpha=0.3)

data_z_cosmo = np.linspace(data_z_hd.min(),data_z_hd.max(),50)
cosmo = FlatLambdaCDM(H0=70, Om0=0.3)
data_mu_cosmo = cosmo.distmod(data_z_cosmo)
ax1.plot(data_z_cosmo, data_mu_cosmo)

plt.xlabel('z')
plt.ylabel('mu')

data_mures = data_mu_hd - cosmo.distmod(data_z_hd).value

ax2 = plt.subplot(3,1,2, sharex=ax1)
ax2.errorbar(data_z_hd, data_mures, yerr=data_mu_err, fmt='.',alpha=0.3)
colors = plt.rcParams["axes.prop_cycle"].by_key()["color"]
ax2.axhline(y=0,color=colors[1])
plt.ylabel('mures')

bins= np.linspace(data_z_hd.min(),data_z_hd.max(),20)
data_mures_mean, bin_edges, _ = stats.binned_statistic(
    data_z_hd, data_mures, statistic='mean', bins= bins
)

data_mures_std, bin_edges, _ = stats.binned_statistic(
    data_z_hd, data_mures, statistic='std', bins= bins
)

data_mures_count, bin_edges, _ = stats.binned_statistic(
    data_z_hd, data_mures, statistic='count', bins= bins
) 

data_mures_err = data_mures_std / np.sqrt(data_mures_count)

bin_mean = (bin_edges[:-1] + bin_edges[1:])*0.5
ax2.errorbar(bin_mean, data_mures_mean, yerr = data_mures_err,fmt='o')

ax3 = plt.subplot(3,1,3, sharex=ax1)
ax3.errorbar(bin_mean, data_mures_mean, yerr = data_mures_err,fmt='o')
ax3.axhline(y=0,color=colors[1])

plt.xlim((0,0.2))

In [ ]:
bins = np.linspace(-1,1,20)
mures.hist(bins=bins,alpha=0.3)
data_mures.hist(bins=bins,alpha=0.3)

In [ ]:
zcut = saltpars["z"]<0.06
hostmass_plot = np.array(hostmass)[saltpar_cuts & zcut]

ax1 = plt.subplot(2,1,1)

ax1.plot(hostmass_plot,mures[zcut],'.')

bins= np.linspace(7,hostmass_plot.max(),10)
hostmass_mean, bin_edges, _ = stats.binned_statistic(
    hostmass_plot, mures[zcut], statistic='mean', bins= bins
)

hostmass_std, bin_edges, _ = stats.binned_statistic(
    hostmass_plot, mures[zcut], statistic='std', bins= bins
)

hostmass_count, bin_edges, _ = stats.binned_statistic(
    hostmass_plot, mures[zcut], statistic='count', bins= bins
) 

hostmass_err = hostmass_std / np.sqrt(hostmass_count)

bin_mean = (bin_edges[:-1] + bin_edges[1:])*0.5
plt.errorbar(bin_mean, hostmass_mean, yerr =hostmass_err,fmt='o')

plt.axhline(y=0,color=colors[1])

plt.xlabel('Hostmass')
plt.ylabel('mures')

ax2 = plt.subplot(2,1,2, sharex=ax1)
plt.errorbar(bin_mean, hostmass_mean, yerr =hostmass_err,fmt='o')

plt.axhline(y=0,color=colors[1])

plt.xlabel('Hostmass')
plt.ylabel('mures')

In [ ]:
hostmass_plot = np.array(hostmass)[saltpar_cuts]

ax1 = plt.subplot(2,1,1)

ax1.plot(hostmass_plot,mures,'.')

bins= np.linspace(7,hostmass_plot.max(),10)
hostmass_mean, bin_edges, _ = stats.binned_statistic(
    hostmass_plot, mures, statistic='mean', bins= bins
)

hostmass_std, bin_edges, _ = stats.binned_statistic(
    hostmass_plot, mures, statistic='std', bins= bins
)

hostmass_count, bin_edges, _ = stats.binned_statistic(
    hostmass_plot, mures, statistic='count', bins= bins
) 

hostmass_err = hostmass_std / np.sqrt(hostmass_count)

bin_mean = (bin_edges[:-1] + bin_edges[1:])*0.5
plt.errorbar(bin_mean, hostmass_mean, yerr =hostmass_err,fmt='o')

plt.axhline(y=0,color=colors[1])

plt.xlabel('Hostmass')
plt.ylabel('mures')

ax2 = plt.subplot(2,1,2, sharex=ax1)
plt.errorbar(bin_mean, hostmass_mean, yerr =hostmass_err,fmt='o')

plt.axhline(y=0,color=colors[1])

plt.xlabel('Hostmass')
plt.ylabel('mures')

In [ ]:
zcut = saltpars["z"]<0.06

x1_plot = np.array(x1)[saltpar_cuts & zcut]

ax1 = plt.subplot(2,1,1)

ax1.plot(x1_plot,mures[zcut],'.')

bins= np.linspace(x1_plot.min(),x1_plot.max(),10)
x1_mean, bin_edges, _ = stats.binned_statistic(
    x1_plot, mures[zcut], statistic='mean', bins= bins
)

x1_std, bin_edges, _ = stats.binned_statistic(
    x1_plot, mures[zcut], statistic='std', bins= bins
)

x1_count, bin_edges, _ = stats.binned_statistic(
    x1_plot, mures[zcut], statistic='count', bins= bins
) 

x1_err = x1_std / np.sqrt(x1_count)

bin_mean = (bin_edges[:-1] + bin_edges[1:])*0.5
plt.errorbar(bin_mean, x1_mean, yerr =x1_err,fmt='o')

plt.axhline(y=0,color=colors[1])

plt.xlabel('x1')
plt.ylabel('mures')

ax2 = plt.subplot(2,1,2, sharex=ax1)
plt.errorbar(bin_mean, x1_mean, yerr =x1_err,fmt='o')

plt.axhline(y=0,color=colors[1])

plt.xlabel('x1')
plt.ylabel('mures')

In [ ]:
x1_plot = np.array(x1)[saltpar_cuts]

ax1 = plt.subplot(2,1,1)

ax1.plot(x1_plot,mures,'.')

bins= np.linspace(x1_plot.min(),x1_plot.max(),10)
x1_mean, bin_edges, _ = stats.binned_statistic(
    x1_plot, mures, statistic='mean', bins= bins
)

x1_std, bin_edges, _ = stats.binned_statistic(
    x1_plot, mures, statistic='std', bins= bins
)

x1_count, bin_edges, _ = stats.binned_statistic(
    x1_plot, mures, statistic='count', bins= bins
) 

x1_err = x1_std / np.sqrt(x1_count)

bin_mean = (bin_edges[:-1] + bin_edges[1:])*0.5
plt.errorbar(bin_mean, x1_mean, yerr =x1_err,fmt='o')

plt.axhline(y=0,color=colors[1])

plt.xlabel('x1')
plt.ylabel('mures')

ax2 = plt.subplot(2,1,2, sharex=ax1)
plt.errorbar(bin_mean, x1_mean, yerr =x1_err,fmt='o')

plt.axhline(y=0,color=colors[1])

plt.xlabel('x1')
plt.ylabel('mures')

In [ ]:
zcut = saltpars["z"]<0.06

c_plot = np.array(c)[saltpar_cuts & zcut]

ax1 = plt.subplot(2,1,1)

ax1.plot(c_plot,mures[zcut],'.')

bins= np.linspace(c_plot.min(),c_plot.max(),10)
c_mean, bin_edges, _ = stats.binned_statistic(
    c_plot, mures[zcut], statistic='mean', bins= bins
)

c_std, bin_edges, _ = stats.binned_statistic(
    c_plot, mures[zcut], statistic='std', bins= bins
)

c_count, bin_edges, _ = stats.binned_statistic(
    c_plot, mures[zcut], statistic='count', bins= bins
) 

c_err = c_std / np.sqrt(c_count)

bin_mean = (bin_edges[:-1] + bin_edges[1:])*0.5
plt.errorbar(bin_mean, c_mean, yerr =c_err,fmt='o')

plt.axhline(y=0,color=colors[1])

plt.xlabel('c')
plt.ylabel('mures')

ax2 = plt.subplot(2,1,2, sharex=ax1)
plt.errorbar(bin_mean, c_mean, yerr =c_err,fmt='o')

plt.axhline(y=0,color=colors[1])

plt.xlabel('c')
plt.ylabel('mures')

In [ ]:
c_plot = np.array(c)[saltpar_cuts]

ax1 = plt.subplot(2,1,1)

ax1.plot(c_plot,mures,'.')

bins= np.linspace(c_plot.min(),c_plot.max(),10)
c_mean, bin_edges, _ = stats.binned_statistic(
    c_plot, mures, statistic='mean', bins= bins
)

c_std, bin_edges, _ = stats.binned_statistic(
    c_plot, mures, statistic='std', bins= bins
)

c_count, bin_edges, _ = stats.binned_statistic(
    c_plot, mures, statistic='count', bins= bins
) 

c_err = c_std / np.sqrt(c_count)

bin_mean = (bin_edges[:-1] + bin_edges[1:])*0.5
plt.errorbar(bin_mean, c_mean, yerr =c_err,fmt='o')

plt.axhline(y=0,color=colors[1])

plt.xlabel('c')
plt.ylabel('mures')

ax2 = plt.subplot(2,1,2, sharex=ax1)
plt.errorbar(bin_mean, c_mean, yerr =c_err,fmt='o')

plt.axhline(y=0,color=colors[1])

plt.xlabel('c')
plt.ylabel('mures')

Effects maybe simulated

ZTF Photometry "pocket effect" (Fig 2 of ZTF DR2 Overview paper https://arxiv.org/pdf/2409.04346)
<img src="figs/pocket_effect.png" width="400" height="300">
